# Recurrente neurale netwerken

In de vorige module hebben we rijke semantische representaties van tekst behandeld. De architectuur die we gebruikten, legt de geaggregeerde betekenis van woorden in een zin vast, maar houdt geen rekening met de **volgorde** van de woorden, omdat de aggregatieoperatie die op de embeddings volgt deze informatie uit de originele tekst verwijdert. Omdat deze modellen geen woordvolgorde kunnen representeren, kunnen ze geen complexere of ambiguere taken oplossen zoals tekstgeneratie of vraagbeantwoording.

Om de betekenis van een tekstvolgorde vast te leggen, gebruiken we een neurale netwerkarchitectuur die **recurrent neural network**, of RNN, wordt genoemd. Bij het gebruik van een RNN voeren we onze zin token voor token door het netwerk, en het netwerk produceert een bepaalde **toestand**, die we vervolgens samen met het volgende token opnieuw aan het netwerk voeren.

![Afbeelding die een voorbeeld van een recurrent neurale netwerk generatieschema toont.](../../../../../translated_images/nl/rnn.27f5c29c53d727b5.webp)

Gegeven de invoerreeks tokens $X_0,\dots,X_n$, creëert de RNN een reeks neurale netwerkblokken en traint deze reeks end-to-end met behulp van backpropagation. Elk netwerkblok ontvangt een paar $(X_i,S_i)$ als invoer en produceert $S_{i+1}$ als resultaat. De laatste toestand $S_n$ of output $Y_n$ wordt ingevoerd in een lineaire classifier om het resultaat te produceren. Alle netwerkblokken delen dezelfde gewichten en worden end-to-end getraind met één backpropagatie-pas.

> De figuur hierboven toont het recurrente neurale netwerk in de ontvouwde vorm (aan de linkerkant), en in een compactere recurrente representatie (aan de rechterkant). Het is belangrijk te beseffen dat alle RNN-cellen dezelfde **deelbare gewichten** hebben.

Omdat toestandsvectoren $S_0,\dots,S_n$ door het netwerk worden doorgegeven, kan de RNN sequentiële afhankelijkheden tussen woorden leren. Bijvoorbeeld, wanneer het woord *niet* ergens in de reeks voorkomt, kan het leren bepaalde elementen binnen de toestandsvector te ontkrachten.

Elke RNN-cel bevat twee gewichtsmatrices: $W_H$ en $W_I$, en bias $b$. Bij elke stap van de RNN wordt, gegeven invoer $X_i$ en invoer-toestand $S_i$, de uitvoer-toestand berekend als $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, waarbij $f$ een activatiefunctie is (vaak $\tanh$).

> Voor problemen zoals tekstgeneratie (die we in de volgende eenheid behandelen) of machinevertaling willen we ook bij elke stap van de RNN een outputwaarde verkrijgen. In dat geval is er ook een andere matrix $W_O$, en wordt de output berekend als $Y_i=f(W_O\times S_i+b_O)$.

Laten we zien hoe recurrente neurale netwerken ons kunnen helpen bij het classificeren van onze nieuwsgegevensset.

> Voor de sandbox-omgeving moeten we de volgende cel uitvoeren om zeker te zijn dat de benodigde bibliotheek is geïnstalleerd en de data is voorgehaald. Als je lokaal werkt, kun je de volgende cel overslaan.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Bij het trainen van grote modellen kan het toewijzen van GPU-geheugen een probleem worden. We moeten ook misschien experimenteren met verschillende minibatch-groottes, zodat de data in ons GPU-geheugen past, maar de training toch snel genoeg is. Als je deze code op je eigen GPU-machine uitvoert, kun je experimenteren met het aanpassen van de minibatch-grootte om de training te versnellen.

> **Opmerking**: Bepaalde versies van NVidia-stuurprogramma's staan erom bekend dat ze het geheugen niet vrijgeven nadat het model is getraind. We draaien meerdere voorbeelden in deze notebooks, en dit kan ervoor zorgen dat het geheugen in bepaalde setups uitgeput raakt, vooral als je je eigen experimenten doet als onderdeel van dezelfde notebook. Als je vreemde fouten tegenkomt bij het starten van de training van het model, kun je overwegen de notebook-kernel opnieuw te starten.


In [3]:
batch_size = 16
embed_size = 64

## Eenvoudige RNN-classificator

In het geval van een eenvoudige RNN is elke terugkerende eenheid een eenvoudig lineair netwerk, dat een invoervektor en een toestandsvektor ontvangt en een nieuwe toestandsvektor produceert. In Keras kan dit worden weergegeven door de `SimpleRNN` laag.

Hoewel we one-hot gecodeerde tokens direct aan de RNN-laag kunnen doorgeven, is dit geen goed idee vanwege hun hoge dimensionaliteit. Daarom gebruiken we een embeddinglaag om de dimensionaliteit van woordvectoren te verlagen, gevolgd door een RNN-laag, en tenslotte een `Dense` classificator.

> **Opmerking**: In situaties waarbij de dimensionaliteit niet zo hoog is, bijvoorbeeld bij tokenisatie op karakterniveau, kan het zinvol zijn om een one-hot gecodeerde token direct aan de RNN-cel door te geven.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Opmerking:** We gebruiken hier een niet-getraagde embeddinglaag voor eenvoud, maar voor betere resultaten kunnen we een voorgetrainde embeddinglaag gebruiken met Word2Vec, zoals beschreven in de vorige unit. Het zou een goede oefening voor je zijn om deze code aan te passen zodat deze werkt met voorgetrainde embeddings.

Laten we nu onze RNN trainen. RNN's zijn over het algemeen vrij moeilijk te trainen, omdat zodra de RNN-cellen worden uitgerold over de sequentielengte, het resulterende aantal lagen dat betrokken is bij backpropagation behoorlijk groot is. Daarom moeten we een kleinere leersnelheid kiezen en het netwerk trainen op een grotere dataset om goede resultaten te krijgen. Dit kan behoorlijk wat tijd kosten, dus het gebruik van een GPU heeft de voorkeur.

Om het proces te versnellen, zullen we het RNN-model alleen trainen op nieuwstitels, zonder de beschrijving. Je kunt proberen te trainen met de beschrijving en kijken of je het model aan de praat krijgt.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Opmerking** dat de nauwkeurigheid hier waarschijnlijk lager zal zijn, omdat we alleen trainen op nieuwstitels.


## Herziening van variabele reeksen

Onthoud dat de `TextVectorization` laag automatisch sequenties met variabele lengte in een minibatch opvult met pad-symbolen. Het blijkt dat die symbolen ook deelnemen aan de training, en ze kunnen de convergentie van het model bemoeilijken.

Er zijn verschillende benaderingen die we kunnen nemen om de hoeveelheid padding te minimaliseren. Een daarvan is om de dataset te herschikken op volgorde van sequentielengte en alle sequenties per grootte te groeperen. Dit kan gedaan worden met de `tf.data.experimental.bucket_by_sequence_length` functie (zie [documentatie](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)).

Een andere benadering is het gebruik van **maskering**. In Keras ondersteunen sommige lagen extra input die aangeeft welke tokens in aanmerking genomen moeten worden tijdens het trainen. Om maskering in ons model op te nemen, kunnen we ofwel een aparte `Masking` laag toevoegen ([docs](https://keras.io/api/layers/core_layers/masking/)), of we kunnen de `mask_zero=True` parameter van onze `Embedding` laag specificeren.

> **Opmerking**: Deze training duurt ongeveer 5 minuten om één epoch te voltooien over de hele dataset. Voel je vrij om de training op elk moment te onderbreken als je geen geduld meer hebt. Wat je ook kunt doen is de hoeveelheid data die wordt gebruikt voor training beperken, door een `.take(...)` clausule toe te voegen na de `ds_train` en `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Nu we masking gebruiken, kunnen we het model trainen op de hele dataset van titels en beschrijvingen.

> **Opmerking**: Is het je opgevallen dat we een vectorizer gebruiken die getraind is op de nieuwstitels, en niet op de hele inhoud van het artikel? Mogelijk kunnen hierdoor sommige tokens worden genegeerd, dus het is beter om de vectorizer opnieuw te trainen. Dit zal echter waarschijnlijk maar een klein effect hebben, dus we blijven voor de eenvoud bij de eerder getrainde vectorizer.


## LSTM: Long short-term memory

Een van de belangrijkste problemen van RNN's is **verdrinkende gradiënten**. RNN's kunnen behoorlijk lang zijn, en kunnen moeite hebben om de gradiënten helemaal terug te propagateren naar de eerste laag van het netwerk tijdens backpropagation. Wanneer dit gebeurt, kan het netwerk geen relaties leren tussen ver uit elkaar liggende tokens. Een manier om dit probleem te vermijden is door **expliciet toestandsbeheer** te introduceren met behulp van **poorten**. De twee meest voorkomende architecturen die poorten introduceren zijn **long short-term memory** (LSTM) en **gated relay unit** (GRU). Hier behandelen we LSTMs.

![Afbeelding die een voorbeeld van een long short term memory-cel toont](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Een LSTM-netwerk is georganiseerd op een vergelijkbare manier als een RNN, maar er zijn twee toestanden die van laag naar laag worden doorgegeven: de actuele toestand $c$, en de verborgen vector $h$. Bij elke eenheid wordt de verborgen vector $h_{t-1}$ gecombineerd met invoer $x_t$, en samen bepalen ze wat er gebeurt met de toestand $c_t$ en uitvoer $h_{t}$ via **poorten**. Elke poort heeft een sigmoidactivatie (uitvoer in het bereik $[0,1]$), die kan worden opgevat als een bitmasker wanneer deze wordt vermenigvuldigd met de toestandsvector. LSTMs hebben de volgende poorten (van links naar rechts op de bovenstaande afbeelding):
* **vergeetpoort** die bepaalt welke componenten van de vector $c_{t-1}$ we moeten vergeten en welke we moeten doorgeven. 
* **invoertoort** die bepaalt hoeveel informatie van de invoervector en de vorige verborgen vector in de toestandsvector moet worden opgenomen.
* **uitvoerpoort** die de nieuwe toestandsvector neemt en beslist welke van de componenten worden gebruikt om de nieuwe verborgen vector $h_t$ te produceren.

De componenten van de toestand $c$ kunnen worden gezien als vlaggen die aan of uit gezet kunnen worden. Bijvoorbeeld, wanneer we de naam *Alice* tegenkomen in de reeks, raden we dat het verwijst naar een vrouw, en zetten we de vlag aan in de toestand die aangeeft dat we een vrouwelijk zelfstandig naamwoord in de zin hebben. Wanneer we verder de woorden *and Tom* tegenkomen, zetten we de vlag aan die aangeeft dat we een meervoudig zelfstandig naamwoord hebben. Door op deze manier de toestand te manipuleren kunnen we de grammaticale eigenschappen van de zin bijhouden.

> **Opmerking**: Hier is een geweldige bron om de interne werking van LSTMs te begrijpen: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) door Christopher Olah.

Hoewel de interne structuur van een LSTM-cel complex kan lijken, verbergt Keras deze implementatie binnen de `LSTM` laag, dus het enige wat we in het bovenstaande voorbeeld hoeven te doen is de recurrente laag te vervangen:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Let op** dat het trainen van LSTMs ook vrij traag is, en je in het begin van het trainen misschien niet veel verbetering in nauwkeurigheid ziet. Je moet mogelijk nog enige tijd doorgaan met trainen om een goede nauwkeurigheid te bereiken.


## Bidirectionele en meerlaagse RNN's

In onze voorbeelden tot nu toe werken de recurrente netwerken vanaf het begin van een reeks tot het einde. Dit voelt natuurlijk voor ons omdat het dezelfde richting volgt als waarin we lezen of naar spraak luisteren. Voor scenario's die echter willekeurige toegang tot de invoerreeks vereisen, is het zinvoller om de recurrente berekening in beide richtingen uit te voeren. RNN's die berekeningen in beide richtingen toelaten worden **bidirectionele** RNN's genoemd, en ze kunnen worden gemaakt door de recurrente laag in te wikkelen met een speciale `Bidirectonal` laag.

> **Opmerking**: De `Bidirectional` laag maakt twee kopieën van de laag erin, en zet de `go_backwards` eigenschap van een van die kopieën op `True`, waardoor het in de tegenovergestelde richting langs de reeks gaat.

Recurrente netwerken, unidirectioneel of bidirectioneel, vangen patronen binnen een reeks op en slaan die op in toestandsvectoren of geven ze terug als output. Net zoals bij convolutionele netwerken kunnen we nog een recurrente laag bouwen na de eerste om hogere patroonniveaus op te vangen, opgebouwd uit lagere patroonniveaus die zijn geëxtraheerd door de eerste laag. Dit leidt ons naar het begrip van een **meerlaagse RNN**, die bestaat uit twee of meer recurrente netwerken waarbij de output van de vorige laag wordt doorgegeven aan de volgende laag als input.

![Afbeelding die een meerlaagse long-short-term-memory-RNN laat zien](../../../../../translated_images/nl/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Afbeelding van [dit prachtige bericht](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) door Fernando López.*

Keras maakt het bouwen van deze netwerken een eenvoudige taak, omdat je simpelweg meer recurrente lagen aan het model toevoegt. Voor alle lagen behalve de laatste moeten we de `return_sequences=True` parameter specificeren, omdat we willen dat de laag alle tussentijdse toestanden teruggeeft, en niet alleen de uiteindelijke toestand van de recurrente berekening.

Laten we een twee-laags bidirectionele LSTM bouwen voor ons classificatieprobleem.

> **Opmerking** deze code kost weer behoorlijk wat tijd om te voltooien, maar het geeft ons de hoogste nauwkeurigheid die we tot nu toe hebben gezien. Dus misschien is het de moeite waard om te wachten en het resultaat te bekijken.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN's voor andere taken

Tot nu toe hebben we ons gericht op het gebruik van RNN's om tekstreeksen te classificeren. Maar ze kunnen veel meer taken aan, zoals tekstgeneratie en machinale vertaling &mdash; we zullen die taken in de volgende eenheid behandelen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
